## 5. ERA5-Land ARCO configuration

In [1]:
import os
import re
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from ecmwf.datastores import Client

In [2]:
# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------

OUTPUT_DIR = Path("Data/Global/era5_backend")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# These should already exist from your previous notebook section.
# If not, Cell 2 reloads them.

SAMPLE_INDEX_PATH = Path(
    "Data/Global/pipeline_stage1/station_date_candidates.parquet"
)

STATION_PATH = Path(
    "Data/Global/pipeline_stage1/stations.parquet"
)

# Requested period is derived from the actual sample table,
# but these are useful sanity boundaries.
MIN_EXPECTED_DATE = pd.Timestamp("2015-04-01")
MAX_EXPECTED_DATE = pd.Timestamp("2026-10-03")

# API definition inherited from the previous Spain project.
API_DECAY = 0.85

# ERA5-Land native grid is 0.1 degree.
ERA5_GRID_DEG = 0.1

print("Configuration ready.")

Configuration ready.


In [3]:
# Load only columns used by the downstream ERA5 workflow. The complete
# candidate index remains on disk in Parquet.
SAMPLE_COLUMNS = [
    "sample_id",
    "station_id",
    "sample_date",
    "latitude",
    "longitude",
]
samples = pd.read_parquet(SAMPLE_INDEX_PATH, columns=SAMPLE_COLUMNS)

if "stations" not in globals():
    stations = pd.read_parquet(STATION_PATH)

samples["sample_date"] = pd.to_datetime(samples["sample_date"]).dt.normalize()

print(f"Samples:  {len(samples):,}")
print(f"Stations: {stations['station_id'].nunique():,}")

display(samples.head())
display(stations.head())

Samples:  3,514,643
Stations: 2,210


,sample_id,station_id,sample_date,latitude,longitude
0,0,AMMA-CATCH__Banizoumbou,2015-04-01,13.5325,2.6604
1,1,AMMA-CATCH__Banizoumbou,2015-04-02,13.5325,2.6604
2,2,AMMA-CATCH__Banizoumbou,2015-04-03,13.5325,2.6604
3,3,AMMA-CATCH__Banizoumbou,2015-04-04,13.5325,2.6604
4,4,AMMA-CATCH__Banizoumbou,2015-04-05,13.5325,2.6604


,station_index,station_id,network,network_name,station,latitude,longitude,elevation_m,climate_zone
0,0,AMMA-CATCH__Banizoumbou,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.53250,2.66040,208.0,BSh
1,1,AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH,AMMA-CATCH,Belefoungou-Mid,9.79506,1.70994,414.0,Aw
2,2,AMMA-CATCH__Belefoungou-Top,AMMA-CATCH,AMMA-CATCH,Belefoungou-Top,9.78986,1.70994,414.0,Aw
3,3,AMMA-CATCH__Nalohou-Mid,AMMA-CATCH,AMMA-CATCH,Nalohou-Mid,9.74530,1.60530,441.0,Aw
4,4,AMMA-CATCH__Nalohou-Top,AMMA-CATCH,AMMA-CATCH,Nalohou-Top,9.74407,1.60580,445.0,Aw


In [4]:
print("Sample date range:")
print("  min:", samples["sample_date"].min())
print("  max:", samples["sample_date"].max())

print("\nUnique sample dates:", samples["sample_date"].nunique())
print("Unique stations:", samples["station_id"].nunique())

assert samples["sample_date"].min() >= MIN_EXPECTED_DATE
assert samples["sample_date"].max() <= MAX_EXPECTED_DATE

Sample date range:
  min: 2015-04-01 00:00:00
  max: 2026-10-03 00:00:00

Unique sample dates: 4204
Unique stations: 2210


In [5]:
def snap_to_era5_grid(values):
    """
    Snap geographic coordinates to the nominal 0.1-degree ERA5-Land grid.
    """
    return np.round(np.asarray(values, dtype=np.float64) / ERA5_GRID_DEG) * ERA5_GRID_DEG


samples["era5_lat"] = snap_to_era5_grid(samples["latitude"])
samples["era5_lon"] = snap_to_era5_grid(samples["longitude"])

samples["era5_lat"] = samples["era5_lat"].round(1)
samples["era5_lon"] = samples["era5_lon"].round(1)

display(
    samples[
        [
            "station_id",
            "latitude",
            "longitude",
            "era5_lat",
            "era5_lon",
        ]
    ].drop_duplicates("station_id").head(20)
)

,station_id,latitude,longitude,era5_lat,era5_lon
0,AMMA-CATCH__Banizoumbou,13.53250,2.66040,13.5,2.7
1371,AMMA-CATCH__Belefoungou-Mid,9.79506,1.70994,9.8,1.7
2316,AMMA-CATCH__Belefoungou-Top,9.78986,1.70994,9.8,1.7
3322,AMMA-CATCH__Nalohou-Mid,9.74530,1.60530,9.7,1.6
4086,AMMA-CATCH__Nalohou-Top,9.74407,1.60580,9.7,1.6
4996,AMMA-CATCH__Tondikiboro,13.54800,2.69560,13.5,2.7
6363,AMMA-CATCH__Wankama,13.64560,2.63200,13.6,2.6
7734,ARM__Anthony,37.21340,-98.09690,37.2,-98.1
9976,ARM__Ashton,37.13300,-97.26600,37.1,-97.3
12823,ARM__Byron,36.88100,-98.28500,36.9,-98.3


In [6]:
era5_points = (
    samples[
        ["era5_lat", "era5_lon"]
    ]
    .drop_duplicates()
    .sort_values(["era5_lat", "era5_lon"])
    .reset_index(drop=True)
)

era5_points.insert(
    0,
    "era5_point_id",
    np.arange(len(era5_points), dtype=np.int32)
)

print(f"Unique ISMN stations:       {samples['station_id'].nunique():,}")
print(f"Unique ERA5 grid cells:     {len(era5_points):,}")

display(era5_points.head())

Unique ISMN stations:       2,210
Unique ERA5 grid cells:     1,489


,era5_point_id,era5_lat,era5_lon
0,0,-45.3,-71.8
1,1,-41.7,147.9
2,2,-41.3,147.8
3,3,-40.9,-73.1
4,4,-40.8,175.9


In [7]:
samples = samples.merge(
    era5_points,
    on=["era5_lat", "era5_lon"],
    how="left",
    validate="many_to_one",
)

assert samples["era5_point_id"].notna().all()

samples["era5_point_id"] = samples["era5_point_id"].astype(np.int32)

display(
    samples[
        [
            "sample_id",
            "station_id",
            "sample_date",
            "era5_point_id",
            "era5_lat",
            "era5_lon",
        ]
    ].head()
)

,sample_id,station_id,sample_date,era5_point_id,era5_lat,era5_lon
0,0,AMMA-CATCH__Banizoumbou,2015-04-01,88,13.5,2.7
1,1,AMMA-CATCH__Banizoumbou,2015-04-02,88,13.5,2.7
2,2,AMMA-CATCH__Banizoumbou,2015-04-03,88,13.5,2.7
3,3,AMMA-CATCH__Banizoumbou,2015-04-04,88,13.5,2.7
4,4,AMMA-CATCH__Banizoumbou,2015-04-05,88,13.5,2.7


In [8]:
ERA5_POINT_INDEX_PATH = OUTPUT_DIR / "era5_point_index.parquet"

era5_points.to_parquet(
    ERA5_POINT_INDEX_PATH,
    index=False,
)

print("Saved:", ERA5_POINT_INDEX_PATH)

Saved: Data/Global/era5_backend/era5_point_index.parquet


In [9]:
client = Client()

auth_result = client.check_authentication()

print("ECMWF authentication successful.")
print(auth_result)

ECMWF authentication successful.
{'id': 587847, 'sub': '2ac68889-5787-47c7-b0b7-1058356b16f2', 'role': None, 'email': 'mayurkunnitigi.matada.phy23@itbhu.ac.in'}


In [10]:
COLLECTION_ID = "reanalysis-era5-land-timeseries"

collection = client.get_collection(COLLECTION_ID)

print("Collection ID:")
print(collection.id)

print("\nTitle:")
print(collection.title)

print("\nTemporal coverage:")
print(collection.begin_datetime)
print(collection.end_datetime)

print("\nBounding box:")
print(collection.bbox)

Collection ID:
reanalysis-era5-land-timeseries

Title:
ERA5 Land hourly time-series data from 1950 to present

Temporal coverage:
1950-01-01 00:00:00+00:00
2026-12-31 00:00:00+00:00

Bounding box:
(-180.0, -89.0, 180.0, 89.0)


In [11]:
constraint_probe = client.apply_constraints(
    COLLECTION_ID,
    {}
)

print(json.dumps(constraint_probe, indent=2, default=str))

{
  "variable": [
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_dewpoint_temperature",
    "2m_temperature",
    "skin_temperature",
    "snow_cover",
    "snow_depth",
    "snowfall",
    "soil_temperature_level_1",
    "soil_temperature_level_2",
    "soil_temperature_level_3",
    "soil_temperature_level_4",
    "surface_pressure",
    "surface_solar_radiation_downwards",
    "surface_thermal_radiation_downwards",
    "total_precipitation",
    "volumetric_soil_water_level_1",
    "volumetric_soil_water_level_2",
    "volumetric_soil_water_level_3",
    "volumetric_soil_water_level_4"
  ],
  "date": [
    "1950-01-02/2026-10-01"
  ],
  "data_format": [
    "csv",
    "netcdf"
  ],
  "area": []
}


In [12]:
ECMWF_KEY = os.getenv("ECMWF_DATASTORES_KEY")

if ECMWF_KEY is None:
    print("ECMWF_DATASTORES_KEY is not set in the environment.")
    print("If your Client uses ~/.ecmwfdatastoresrc, run Cell 12.")
else:
    print("ECMWF_DATASTORES_KEY found.")

ECMWF_DATASTORES_KEY is not set in the environment.
If your Client uses ~/.ecmwfdatastoresrc, run Cell 12.


In [13]:
def read_ecmwf_datastores_key():
    """
    Read the key from ~/.ecmwfdatastoresrc if it isn't in the environment.

    Expected format:

        url: https://cds.climate.copernicus.eu/api
        key: YOUR_KEY
    """
    config_path = Path("~/.ecmwfdatastoresrc").expanduser()

    if not config_path.exists():
        return None

    text = config_path.read_text(encoding="utf-8")

    match = re.search(
        r"^\s*key\s*:\s*(\S+)\s*$",
        text,
        flags=re.MULTILINE,
    )

    if match:
        return match.group(1).strip()

    return None


if ECMWF_KEY is None:
    ECMWF_KEY = read_ecmwf_datastores_key()

if ECMWF_KEY is None:
    raise RuntimeError(
        "Could not obtain ECMWF API key. "
        "Set ECMWF_DATASTORES_KEY or configure ~/.ecmwfdatastoresrc."
    )

print("ECMWF API key available.")

ECMWF API key available.


In [14]:
T2M_D2M_ZARR = (
    "https://arco.datastores.ecmwf.int/"
    "cadl-arco-geo-007/arco/reanalysis_era5_land/"
    "sfc-2m-temperature/geoChunked.zarr"
)

PRECIP_ZARR = (
    "https://arco.datastores.ecmwf.int/"
    "cadl-arco-geo-009/arco/reanalysis_era5_land/"
    "sfc-pressure-precipitation/geoChunked.zarr"
)

print(T2M_D2M_ZARR)
print(PRECIP_ZARR)

https://arco.datastores.ecmwf.int/cadl-arco-geo-007/arco/reanalysis_era5_land/sfc-2m-temperature/geoChunked.zarr
https://arco.datastores.ecmwf.int/cadl-arco-geo-009/arco/reanalysis_era5_land/sfc-pressure-precipitation/geoChunked.zarr


In [15]:
T2M_D2M_ZARR = (
    "https://arco.datastores.ecmwf.int/"
    "cadl-arco-geo-007/arco/reanalysis_era5_land/"
    "sfc-2m-temperature/geoChunked.zarr"
)

PRECIP_ZARR = (
    "https://arco.datastores.ecmwf.int/"
    "cadl-arco-geo-009/arco/reanalysis_era5_land/"
    "sfc-pressure-precipitation/geoChunked.zarr"
)

print(T2M_D2M_ZARR)
print(PRECIP_ZARR)

https://arco.datastores.ecmwf.int/cadl-arco-geo-007/arco/reanalysis_era5_land/sfc-2m-temperature/geoChunked.zarr
https://arco.datastores.ecmwf.int/cadl-arco-geo-009/arco/reanalysis_era5_land/sfc-pressure-precipitation/geoChunked.zarr


In [16]:
if not client.key:
    raise RuntimeError("ECMWF authentication succeeded but no CDS API key was available for ARCO access.")

storage_options = {
    "headers": {
        "Authorization": f"Bearer {client.key}"
    }
}

print("Opening temperature/dewpoint store...")

era5_met = xr.open_zarr(
    T2M_D2M_ZARR,
    consolidated=True,
    storage_options=storage_options,
)

print(era5_met)

print("\nOpening precipitation store...")

era5_precip = xr.open_zarr(
    PRECIP_ZARR,
    consolidated=True,
    storage_options=storage_options,
)

print(era5_precip)

Opening temperature/dewpoint store...


<xarray.Dataset> Size: 35TB
Dimensions:    (time: 672768, latitude: 1801, longitude: 3600)
Coordinates:
  * time       (time) datetime64[ns] 5MB 1950-01-02 ... 2026-10-01T23:00:00
  * latitude   (latitude) float64 14kB -90.0 -89.9 -89.8 ... 89.8 89.9 90.0
  * longitude  (longitude) float64 29kB -179.9 -179.8 -179.7 ... 179.9 180.0
Data variables:
    d2m        (time, latitude, longitude) float32 17TB dask.array<chunksize=(33792, 4, 8), meta=np.ndarray>
    t2m        (time, latitude, longitude) float32 17TB dask.array<chunksize=(33792, 4, 8), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_edition:            1
    GRIB_subCentre:          0
    history:                 2025-02-10T00:00 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

Opening precipitation store...
<xarray.Dataset>

In [17]:
def inspect_grid(ds, name):
    print(f"\n{name}")
    print("-" * 60)

    print(
        "Latitude:",
        float(ds.latitude.min()),
        "→",
        float(ds.latitude.max()),
    )

    print(
        "Longitude:",
        float(ds.longitude.min()),
        "→",
        float(ds.longitude.max()),
    )

    print("Latitude direction:", ds.latitude.values[1] - ds.latitude.values[0])
    print("Longitude direction:", ds.longitude.values[1] - ds.longitude.values[0])

    print("Variables:", list(ds.data_vars))


inspect_grid(era5_met, "ERA5 temperature/dewpoint")
inspect_grid(era5_precip, "ERA5 precipitation")


ERA5 temperature/dewpoint
------------------------------------------------------------
Latitude: -89.99999999999832 → 90.0
Longitude: -179.90000000000595 → 179.9999999999941
Latitude direction: 0.09999999999999432
Longitude direction: 0.0999999999999659
Variables: ['d2m', 't2m']

ERA5 precipitation
------------------------------------------------------------
Latitude: -89.99999999999832 → 90.0
Longitude: -179.90000000000595 → 179.9999999999941
Latitude direction: 0.09999999999999432
Longitude direction: 0.0999999999999659
Variables: ['sp', 'tp']


In [18]:
REMOTE_LON_MIN = float(era5_met.longitude.min())
REMOTE_LON_MAX = float(era5_met.longitude.max())

def longitude_for_era5(lon):
    """
    Convert [-180, 180] longitude to the convention used by the remote dataset.
    """
    lon = np.asarray(lon, dtype=np.float64)

    if REMOTE_LON_MAX > 180:
        # 0..360 convention
        return lon % 360.0

    # -180..180 convention
    return lon


era5_points["query_lon"] = longitude_for_era5(
    era5_points["era5_lon"].values
)

display(era5_points.head())

,era5_point_id,era5_lat,era5_lon,query_lon
0,0,-45.3,-71.8,-71.8
1,1,-41.7,147.9,147.9
2,2,-41.3,147.8,147.8
3,3,-40.9,-73.1,-73.1
4,4,-40.8,175.9,175.9


In [19]:
print("Meteorological variables:")
print(list(era5_met.data_vars))

print("\nPrecipitation variables:")
print(list(era5_precip.data_vars))

Meteorological variables:
['d2m', 't2m']

Precipitation variables:
['sp', 'tp']


In [20]:
era5_lat_query = xr.DataArray(
    era5_points["era5_lat"].to_numpy(dtype=np.float64),
    dims="era5_point",
    coords={
        "era5_point": era5_points["era5_point_id"].to_numpy()
    },
)

era5_lon_query = xr.DataArray(
    era5_points["query_lon"].to_numpy(dtype=np.float64),
    dims="era5_point",
    coords={
        "era5_point": era5_points["era5_point_id"].to_numpy()
    },
)

print(era5_lat_query)
print(era5_lon_query)

<xarray.DataArray (era5_point: 1489)> Size: 12kB
array([-45.3, -41.7, -41.3, ...,  69. ,  70.3,  71.3], shape=(1489,))
Coordinates:
  * era5_point  (era5_point) int32 6kB 0 1 2 3 4 5 ... 1484 1485 1486 1487 1488
<xarray.DataArray (era5_point: 1489)> Size: 12kB
array([ -71.8,  147.9,  147.8, ...,   19.4, -148.9, -156.6], shape=(1489,))
Coordinates:
  * era5_point  (era5_point) int32 6kB 0 1 2 3 4 5 ... 1484 1485 1486 1487 1488


In [21]:
TEST_YEAR = 2020

year_start = pd.Timestamp(f"{TEST_YEAR}-01-01")
year_end = pd.Timestamp(f"{TEST_YEAR + 1}-01-01")

print(year_start)
print(year_end)

2020-01-01 00:00:00
2021-01-01 00:00:00


In [22]:
met_test = era5_met[["t2m", "d2m"]].sel(
    time=slice(year_start, year_end),
).sel(
    latitude=era5_lat_query,
    longitude=era5_lon_query,
    method="nearest",
)

precip_test = era5_precip[["tp"]].sel(
    time=slice(year_start, year_end),
).sel(
    latitude=era5_lat_query,
    longitude=era5_lon_query,
    method="nearest",
)

print(met_test)
print(precip_test)

<xarray.Dataset> Size: 105MB
Dimensions:     (time: 8785, era5_point: 1489)
Coordinates:
  * time        (time) datetime64[ns] 70kB 2020-01-01 ... 2021-01-01
  * era5_point  (era5_point) int32 6kB 0 1 2 3 4 5 ... 1484 1485 1486 1487 1488
    latitude    (era5_point) float64 12kB -45.3 -41.7 -41.3 ... 69.0 70.3 71.3
    longitude   (era5_point) float64 12kB -71.8 147.9 147.8 ... -148.9 -156.6
Data variables:
    t2m         (time, era5_point) float32 52MB dask.array<chunksize=(8785, 32), meta=np.ndarray>
    d2m         (time, era5_point) float32 52MB dask.array<chunksize=(8785, 32), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_edition:            1
    GRIB_subCentre:          0
    history:                 2025-02-10T00:00 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts
<xarray.

In [23]:
print("Temperature dimensions:")
print(met_test.dims)

print("\nTemperature shape:")
print(met_test["t2m"].shape)

print("\nPrecipitation shape:")
print(precip_test["tp"].shape)

Temperature dimensions:
FrozenMappingWarningOnValuesAccess({'time': 8785, 'era5_point': 1489})

Temperature shape:
(8785, 1489)

Precipitation shape:
(8785, 1489)


In [24]:
def daily_era5_features(met_ds, precip_ds):
    """
    Convert hourly point time series into daily features.

    t2m/d2m:
        daily mean

    tp:
        daily sum

    ERA5-Land tp is expressed in metres.
    """

    t2m_daily = (
        met_ds["t2m"]
        .resample(time="1D")
        .mean()
        .rename("t2m_K")
    )

    d2m_daily = (
        met_ds["d2m"]
        .resample(time="1D")
        .mean()
        .rename("d2m_K")
    )

    precip_daily = (
        precip_ds["tp"]
        .resample(time="1D")
        .sum()
        .rename("precip_m")
    )

    precip_daily_mm = (
        precip_daily * 1000.0
    ).rename("precip_mm")

    return xr.Dataset({
        "t2m_K": t2m_daily,
        "d2m_K": d2m_daily,
        "precip_mm": precip_daily_mm,
    })

In [25]:
def saturation_vapor_pressure_kpa(temp_c):
    return (
        0.6108
        * np.exp(
            17.27 * temp_c / (temp_c + 237.3)
        )
    )


def calculate_vpd_from_dataset(ds):
    t_c = ds["t2m_K"] - 273.15
    td_c = ds["d2m_K"] - 273.15

    es = saturation_vapor_pressure_kpa(t_c)
    ea = saturation_vapor_pressure_kpa(td_c)

    vpd = (es - ea).clip(min=0)

    return vpd.rename("VPD_kPa")

In [26]:
import numpy as np
import xarray as xr

# Define API decay constant if not already defined
API_DECAY = 0.85


def calculate_api_xarray(precip_da, decay=API_DECAY):
  """Calculates Antecedent Precipitation Index (API) continuously across the time dimension.

  API_t = Precip_t + decay * API_(t-1)
  """
  values = precip_da.values  # Expected shape: (time, points)
  api = np.zeros_like(values, dtype=np.float32)

  # Initial step
  api[0] = values[0]

  # Sequential decay calculation across time dimension
  for t in range(1, values.shape[0]):
    api[t] = values[t] + (decay * api[t - 1])

  return xr.DataArray(api, coords=precip_da.coords, dims=precip_da.dims)

In [27]:
import gc
import time
from pathlib import Path
import dask
import pandas as pd
import xarray as xr
from tqdm.notebook import tqdm

BATCH_CACHE_DIR = OUTPUT_DIR / "batch_cache"
BATCH_CACHE_DIR.mkdir(parents=True, exist_ok=True)

full_start = samples["sample_date"].min()
full_end = samples["sample_date"].max()

# -------------------------------------------------------------------
# 1. Identify Already Cached Points (Keep 54% Progress)
# -------------------------------------------------------------------
all_points_df = pd.DataFrame(
    {
        "era5_point": era5_lat_query.era5_point.values,
        "lat": era5_lat_query.values,
        "lon": era5_lon_query.values,
    }
)

cached_files = sorted(BATCH_CACHE_DIR.glob("batch_*.parquet"))
if cached_files:
  print("Scanning existing batch cache to preserve progress...")
  cached_points = set()
  for f in cached_files:
    try:
      df_c = pd.read_parquet(f, columns=["era5_point"])
      cached_points.update(df_c["era5_point"].unique())
    except Exception:
      pass
  print(
      f"Found {len(cached_points):,} unique points already completed on disk."
  )
  remaining_df = all_points_df[
      ~all_points_df["era5_point"].isin(cached_points)
  ].copy()
else:
  remaining_df = all_points_df.copy()

print(f"Remaining points to fetch: {len(remaining_df):,}")

# -------------------------------------------------------------------
# 2. Group Remaining Points into Larger 1.5-Degree Clusters
# -------------------------------------------------------------------
# 1.5 degrees (~160 km) reduces 967 requests down to ~40-60 requests
BIN_SIZE = 1.5
remaining_df["lat_bin"] = (
    remaining_df["lat"] / BIN_SIZE
).round() * BIN_SIZE
remaining_df["lon_bin"] = (
    remaining_df["lon"] / BIN_SIZE
).round() * BIN_SIZE

spatial_groups = list(remaining_df.groupby(["lat_bin", "lon_bin"]))
print(f"Grouped remaining points into {len(spatial_groups)} macro-clusters.")

lat_is_descending = bool(era5_met.latitude[0] > era5_met.latitude[-1])
lon_is_descending = bool(era5_met.longitude[0] > era5_met.longitude[-1])


def get_spatial_slice(min_val, max_val, is_descending, buffer=0.25):
  if is_descending:
    return slice(max_val + buffer, min_val - buffer)
  return slice(min_val - buffer, max_val + buffer)


t0 = time.perf_counter()

# -------------------------------------------------------------------
# 3. Stream Remaining Macro-Clusters (2 Threads + 0.2s Sleep)
# -------------------------------------------------------------------
for batch_idx, ((lat_b, lon_b), sub_df) in enumerate(
    tqdm(spatial_groups, desc="Extracting Remaining Macro-Clusters")
):
  # Use new prefix to avoid overwriting original batch files
  batch_file = BATCH_CACHE_DIR / f"macro_batch_{batch_idx:04d}.parquet"

  if batch_file.exists():
    continue

  batch_lat = xr.DataArray(
      sub_df["lat"].values,
      dims="era5_point",
      coords={"era5_point": sub_df["era5_point"].values},
  )
  batch_lon = xr.DataArray(
      sub_df["lon"].values,
      dims="era5_point",
      coords={"era5_point": sub_df["era5_point"].values},
  )

  lat_sl = get_spatial_slice(
      sub_df["lat"].min(), sub_df["lat"].max(), lat_is_descending, buffer=0.25
  )
  lon_sl = get_spatial_slice(
      sub_df["lon"].min(), sub_df["lon"].max(), lon_is_descending, buffer=0.25
  )

  # Use 2 parallel workers (fast without triggering server connection resets)
  with dask.config.set(scheduler="threads", num_workers=2):
    met_box = (
        era5_met[["t2m", "d2m"]]
        .sel(
            time=slice(full_start, full_end),
            latitude=lat_sl,
            longitude=lon_sl,
        )
        .compute()
    )
    precip_box = (
        era5_precip[["tp"]]
        .sel(
            time=slice(full_start, full_end),
            latitude=lat_sl,
            longitude=lon_sl,
        )
        .compute()
    )

  met_hourly = met_box.sel(
      latitude=batch_lat, longitude=batch_lon, method="nearest"
  )
  precip_hourly = precip_box.sel(
      latitude=batch_lat, longitude=batch_lon, method="nearest"
  )

  daily = daily_era5_features(met_hourly, precip_hourly)
  daily["VPD_kPa"] = calculate_vpd_from_dataset(daily)
  daily["API_mm"] = calculate_api_xarray(daily["precip_mm"], decay=API_DECAY)

  batch_df = daily.to_dataframe().reset_index()
  batch_df.to_parquet(batch_file, index=False)

  del met_box, precip_box, met_hourly, precip_hourly, daily, batch_df
  gc.collect()

  time.sleep(0.2)  # Fast, safe pacing delay

# -------------------------------------------------------------------
# 4. Combine ALL Batches (Original 54% + New Macro Batches)
# -------------------------------------------------------------------
print("\nCombining all cached batch files into final DataFrame...")
all_files = sorted(BATCH_CACHE_DIR.glob("*.parquet"))
era5_daily = pd.concat([pd.read_parquet(f) for f in all_files], ignore_index=True)

elapsed = time.perf_counter() - t0
print(f"Remaining extraction complete in {elapsed / 60:.2f} minutes.")
print(f"Total rows fetched: {len(era5_daily):,}")

Scanning existing batch cache to preserve progress...
Found 888 unique points already completed on disk.
Remaining points to fetch: 601
Grouped remaining points into 242 macro-clusters.


Extracting Remaining Macro-Clusters:   0%|          | 0/242 [00:00<?, ?it/s]


Combining all cached batch files into final DataFrame...
Remaining extraction complete in 320.24 minutes.
Total rows fetched: 6,255,936


In [28]:
daily_test = daily_era5_features(
    met_test,
    precip_test,
)

daily_test["VPD_kPa"] = calculate_vpd_from_dataset(
    daily_test
)

print(daily_test)

<xarray.Dataset> Size: 9MB
Dimensions:     (era5_point: 1489, time: 367)
Coordinates:
  * era5_point  (era5_point) int32 6kB 0 1 2 3 4 5 ... 1484 1485 1486 1487 1488
    latitude    (era5_point) float64 12kB -45.3 -41.7 -41.3 ... 69.0 70.3 71.3
    longitude   (era5_point) float64 12kB -71.8 147.9 147.8 ... -148.9 -156.6
  * time        (time) datetime64[ns] 3kB 2020-01-01 2020-01-02 ... 2021-01-01
Data variables:
    t2m_K       (time, era5_point) float32 2MB dask.array<chunksize=(1, 32), meta=np.ndarray>
    d2m_K       (time, era5_point) float32 2MB dask.array<chunksize=(1, 32), meta=np.ndarray>
    precip_mm   (time, era5_point) float32 2MB dask.array<chunksize=(1, 32), meta=np.ndarray>
    VPD_kPa     (time, era5_point) float32 2MB dask.array<chunksize=(1, 32), meta=np.ndarray>


In [29]:
def calculate_api_xarray(
    precip_mm: xr.DataArray,
    decay: float = 0.85,
):
    """
    Calculate API independently along the time dimension
    for every ERA5 grid point.

    Input shape:
        time × era5_point

    Output:
        same shape
    """

    values = precip_mm.values

    api = np.full_like(
        values,
        np.nan,
        dtype=np.float32,
    )

    api[0, :] = values[0, :]

    for t in range(1, values.shape[0]):

        current = values[t, :]
        previous = api[t - 1, :]

        valid_current = ~np.isnan(current)

        api[t, valid_current] = (
            current[valid_current]
            + decay * np.nan_to_num(
                previous[valid_current],
                nan=0.0,
            )
        )

    return xr.DataArray(
        api,
        coords=precip_mm.coords,
        dims=precip_mm.dims,
        name="API_mm",
    )

In [30]:
daily_test["API_mm"] = calculate_api_xarray(
    daily_test["precip_mm"],
    decay=API_DECAY,
)

print(daily_test)

<xarray.Dataset> Size: 11MB
Dimensions:     (era5_point: 1489, time: 367)
Coordinates:
  * era5_point  (era5_point) int32 6kB 0 1 2 3 4 5 ... 1484 1485 1486 1487 1488
    latitude    (era5_point) float64 12kB -45.3 -41.7 -41.3 ... 69.0 70.3 71.3
    longitude   (era5_point) float64 12kB -71.8 147.9 147.8 ... -148.9 -156.6
  * time        (time) datetime64[ns] 3kB 2020-01-01 2020-01-02 ... 2021-01-01
Data variables:
    t2m_K       (time, era5_point) float32 2MB dask.array<chunksize=(1, 32), meta=np.ndarray>
    d2m_K       (time, era5_point) float32 2MB dask.array<chunksize=(1, 32), meta=np.ndarray>
    precip_mm   (time, era5_point) float32 2MB dask.array<chunksize=(1, 32), meta=np.ndarray>
    VPD_kPa     (time, era5_point) float32 2MB dask.array<chunksize=(1, 32), meta=np.ndarray>
    API_mm      (time, era5_point) float32 2MB 0.002146 0.01168 ... 1.255 1.793


In [31]:
t0 = time.perf_counter()

daily_test_loaded = daily_test.compute()

elapsed = time.perf_counter() - t0

print(f"Remote extraction + computation: {elapsed:.2f} seconds")

display(daily_test_loaded.isel(time=slice(0, 10)))

Remote extraction + computation: 2364.29 seconds


<xarray.Dataset> Size: 328kB
Dimensions:     (era5_point: 1489, time: 10)
Coordinates:
  * era5_point  (era5_point) int32 6kB 0 1 2 3 4 5 ... 1484 1485 1486 1487 1488
    latitude    (era5_point) float64 12kB -45.3 -41.7 -41.3 ... 69.0 70.3 71.3
    longitude   (era5_point) float64 12kB -71.8 147.9 147.8 ... -148.9 -156.6
  * time        (time) datetime64[ns] 80B 2020-01-01 2020-01-02 ... 2020-01-10
Data variables:
    t2m_K       (time, era5_point) float32 60kB 288.4 289.0 ... 238.3 239.9
    d2m_K       (time, era5_point) float32 60kB 277.5 280.4 ... 234.9 235.8
    precip_mm   (time, era5_point) float32 60kB 0.002146 0.01168 ... 0.005275
    VPD_kPa     (time, era5_point) float32 60kB 0.904 0.7733 ... 0.01229
    API_mm      (time, era5_point) float32 60kB 0.002146 0.01168 ... 0.4101

In [32]:
test_df = (
    daily_test_loaded
    .to_dataframe()
    .reset_index()
)

display(test_df.head())

print("Rows:", len(test_df))
print("Columns:", list(test_df.columns))

,era5_point,time,latitude,longitude,t2m_K,d2m_K,precip_mm,VPD_kPa,API_mm
0,0,2020-01-01,-45.3,-71.8,288.414764,277.451263,0.002146,0.903966,0.002146
1,0,2020-01-02,-45.3,-71.8,288.767883,280.868378,0.000876,0.722014,0.002700
2,0,2020-01-03,-45.3,-71.8,286.470367,282.238617,5.297160,0.374498,5.299455
3,0,2020-01-04,-45.3,-71.8,280.797211,274.636475,0.102059,0.367138,4.606596
4,0,2020-01-05,-45.3,-71.8,279.666290,276.323395,5.179741,0.201923,9.095347


Rows: 546463
Columns: ['era5_point', 'time', 'latitude', 'longitude', 't2m_K', 'd2m_K', 'precip_mm', 'VPD_kPa', 'API_mm']


In [33]:
selected_lat = (
    met_test
    .isel(time=0)["latitude"]
    .to_numpy()
)

selected_lon = (
    met_test
    .isel(time=0)["longitude"]
    .to_numpy()
)

remote_point_check = era5_points.copy()

remote_point_check["remote_lat"] = selected_lat
remote_point_check["remote_lon"] = selected_lon

display(remote_point_check.head(20))

,era5_point_id,era5_lat,era5_lon,query_lon,remote_lat,remote_lon
0,0,-45.3,-71.8,-71.8,-45.3,-71.8
1,1,-41.7,147.9,147.9,-41.7,147.9
2,2,-41.3,147.8,147.8,-41.3,147.8
3,3,-40.9,-73.1,-73.1,-40.9,-73.1
4,4,-40.8,175.9,175.9,-40.8,175.9
5,5,-40.7,175.9,175.9,-40.7,175.9
6,6,-36.7,174.6,174.6,-36.7,174.6
7,7,-35.7,148.2,148.2,-35.7,148.2
8,8,-35.6,148.8,148.8,-35.6,148.8
9,9,-35.4,147.5,147.5,-35.4,147.5


In [34]:
def dataset_memory_mb(ds):
    total = 0

    for var in ds.data_vars:
        total += ds[var].nbytes

    return total / (1024 ** 2)


print(
    "Selected hourly temperature/dewpoint:",
    f"{dataset_memory_mb(met_test):.2f} MB"
)

print(
    "Selected hourly precipitation:",
    f"{dataset_memory_mb(precip_test):.2f} MB"
)

print(
    "Daily dataset:",
    f"{dataset_memory_mb(daily_test_loaded):.2f} MB"
)

Selected hourly temperature/dewpoint: 99.80 MB
Selected hourly precipitation: 49.90 MB
Daily dataset: 10.42 MB


In [35]:
sample_min_year = samples["sample_date"].dt.year.min()
sample_max_year = samples["sample_date"].dt.year.max()

YEARS = list(
    range(
        sample_min_year,
        sample_max_year + 1,
    )
)

print(YEARS)

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [36]:
def extract_era5_year(
    year: int,
    era5_met,
    era5_precip,
    lat_query,
    lon_query,
):
    """
    Extract one calendar year's daily ERA5-Land features
    for every unique ERA5 grid point used by ISMN.
    """

    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year + 1}-01-01")

    t0 = time.perf_counter()
    def log(message):
        elapsed = time.perf_counter() - t0
        print(f"[{time.strftime('%H:%M:%S')}] +{elapsed:7.1f}s {message}", flush=True)

    log(f"START year={year}, points={lat_query.size:,}, batch_size=64")
    batch_size = 64
    batches = []

    for batch_start in range(0, lat_query.size, batch_size):
        batch_stop = min(batch_start + batch_size, lat_query.size)
        batch_lat = lat_query.isel(era5_point=slice(batch_start, batch_stop))
        batch_lon = lon_query.isel(era5_point=slice(batch_start, batch_stop))

        log(f"batch {batch_start // batch_size + 1}: selecting remote meteorology points {batch_start}:{batch_stop}")
        phase_t0 = time.perf_counter()
        met = era5_met[["t2m", "d2m"]].sel(
            time=slice(start, end),
        ).sel(
            latitude=batch_lat,
            longitude=batch_lon,
            method="nearest",
        )

        log(f"batch {batch_start // batch_size + 1}: selecting remote precipitation points {batch_start}:{batch_stop}")
        precip = era5_precip[["tp"]].sel(
            time=slice(start, end),
        ).sel(
            latitude=batch_lat,
            longitude=batch_lon,
            method="nearest",
        )

        log(f"batch {batch_start // batch_size + 1}: selections ready in {time.perf_counter() - phase_t0:.1f}s; computing daily features and downloading chunks")
        phase_t0 = time.perf_counter()

        daily = daily_era5_features(met, precip)

        daily["VPD_kPa"] = calculate_vpd_from_dataset(daily)

        daily["API_mm"] = calculate_api_xarray(
            daily["precip_mm"],
            decay=API_DECAY,
        )

        batch_df = daily.compute().to_dataframe().reset_index()
        batches.append(batch_df)
        log(f"batch {batch_start // batch_size + 1}: download/compute/pandas complete in {time.perf_counter() - phase_t0:.1f}s; rows={len(batch_df):,}; points={batch_stop:,}/{lat_query.size:,}")

    df = pd.concat(batches, ignore_index=True)
    df["year"] = year

    elapsed = time.perf_counter() - t0

    log(
        f"Finished {year}: "
        f"{len(df):,} rows in {elapsed:.1f}s"
    )

    return df

In [37]:
year_sample_counts = (
    samples
    .assign(year=samples["sample_date"].dt.year)
    .groupby("year")
    .size()
    .rename("samples")
    .reset_index()
)

display(year_sample_counts)

,year,samples
0,2015,329903
1,2016,394790
2,2017,369953
3,2018,344806
4,2019,308679
5,2020,269853
6,2021,259625
7,2022,260490
8,2023,233307
9,2024,264946


In [38]:
TEST_PRODUCTION_YEAR = YEARS[0]

production_test = extract_era5_year(
    TEST_PRODUCTION_YEAR,
    era5_met,
    era5_precip,
    era5_lat_query,
    era5_lon_query,
)

display(production_test.head())

[04:13:11] +    0.0s START year=2015, points=1,489, batch_size=64
[04:13:11] +    0.0s batch 1: selecting remote meteorology points 0:64
[04:13:13] +    1.2s batch 1: selecting remote precipitation points 0:64
[04:13:13] +    2.0s batch 1: selections ready in 2.0s; computing daily features and downloading chunks


[04:18:38] +  327.0s batch 1: download/compute/pandas complete in 325.0s; rows=23,424; points=64/1,489
[04:18:38] +  327.0s batch 2: selecting remote meteorology points 64:128
[04:18:39] +  328.2s batch 2: selecting remote precipitation points 64:128
[04:18:40] +  328.7s batch 2: selections ready in 1.7s; computing daily features and downloading chunks
[04:23:34] +  622.4s batch 2: download/compute/pandas complete in 293.7s; rows=23,424; points=128/1,489
[04:23:34] +  622.4s batch 3: selecting remote meteorology points 128:192
[04:23:34] +  623.0s batch 3: selecting remote precipitation points 128:192
[04:23:35] +  623.4s batch 3: selections ready in 1.0s; computing daily features and downloading chunks
[04:27:36] +  864.9s batch 3: download/compute/pandas complete in 241.6s; rows=23,424; points=192/1,489
[04:27:36] +  864.9s batch 4: selecting remote meteorology points 192:256
[04:27:37] +  865.6s batch 4: selecting remote precipitation points 192:256
[04:27:37] +  866.0s batch 4: sel

ClientPayloadError: Response payload is not completed: <ContentLengthError: 400, message='Not enough data to satisfy content length header.'>

In [ ]:
test_output = (
    OUTPUT_DIR
    / f"era5_grid_daily_{TEST_PRODUCTION_YEAR}.parquet"
)

production_test.to_parquet(
    test_output,
    index=False,
)

print("Saved:", test_output)

In [39]:
YEARLY_DIR = OUTPUT_DIR / "yearly"
YEARLY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

completed = []

for year in YEARS:

    output_path = (
        YEARLY_DIR
        / f"era5_grid_daily_{year}.parquet"
    )

    if output_path.exists():
        print(f"[SKIP] {year}: already exists")
        completed.append(year)
        continue

    try:
        df_year = extract_era5_year(
            year,
            era5_met,
            era5_precip,
            era5_lat_query,
            era5_lon_query,
        )

        df_year.to_parquet(
            output_path,
            index=False,
        )

        completed.append(year)

        print(
            f"[OK] {year} -> {output_path}"
        )

    except Exception as exc:
        print(
            f"[FAILED] {year}: "
            f"{type(exc).__name__}: {exc}"
        )
        raise

[05:16:42] +    0.0s START year=2015, points=1,489, batch_size=64
[05:16:42] +    0.0s batch 1: selecting remote meteorology points 0:64
[05:16:42] +    0.9s batch 1: selecting remote precipitation points 0:64
[05:16:43] +    1.4s batch 1: selections ready in 1.4s; computing daily features and downloading chunks


KeyboardInterrupt: 

In [ ]:
year_files = sorted(
    YEARLY_DIR.glob("era5_grid_daily_*.parquet")
)

print("Yearly files:", len(year_files))

era5_daily = pd.concat(
    [
        pd.read_parquet(path)
        for path in year_files
    ],
    ignore_index=True,
)

print(
    f"Total ERA5 daily rows: "
    f"{len(era5_daily):,}"
)

display(era5_daily.head())

In [40]:
era5_daily = era5_daily.rename(
    columns={
        "era5_point": "era5_point_id"
    }
)

era5_daily["era5_point_id"] = (
    era5_daily["era5_point_id"]
    .astype(np.int32)
)

era5_daily = era5_daily.merge(
    era5_points[
        [
            "era5_point_id",
            "era5_lat",
            "era5_lon",
        ]
    ],
    on="era5_point_id",
    how="left",
    validate="many_to_one",
)

display(era5_daily.head())

,era5_point_id,time,latitude,longitude,t2m_K,d2m_K,precip_mm,VPD_kPa,API_mm,era5_lat,era5_lon
0,0,2015-04-01,-45.3,-71.8,277.068176,273.993744,37.424908,0.159249,37.424908,-45.3,-71.8
1,0,2015-04-02,-45.3,-71.8,275.266327,273.295319,8.254624,0.094248,40.065796,-45.3,-71.8
2,0,2015-04-03,-45.3,-71.8,277.224976,274.257233,0.661398,0.155754,34.717327,-45.3,-71.8
3,0,2015-04-04,-45.3,-71.8,276.767731,272.226715,0.002491,0.220681,29.512220,-45.3,-71.8
4,0,2015-04-05,-45.3,-71.8,278.288666,272.367340,0.000435,0.303918,25.085823,-45.3,-71.8


In [41]:
era5_daily["time"] = pd.to_datetime(
    era5_daily["time"]
).dt.normalize()

era5_daily = era5_daily.rename(
    columns={"time": "sample_date"}
)

era5_daily["sample_date"] = pd.to_datetime(
    era5_daily["sample_date"]
)

display(
    era5_daily[
        [
            "era5_point_id",
            "era5_lat",
            "era5_lon",
            "sample_date",
            "t2m_K",
            "d2m_K",
            "precip_mm",
            "VPD_kPa",
            "API_mm",
        ]
    ].head()
)

,era5_point_id,era5_lat,era5_lon,sample_date,t2m_K,d2m_K,precip_mm,VPD_kPa,API_mm
0,0,-45.3,-71.8,2015-04-01,277.068176,273.993744,37.424908,0.159249,37.424908
1,0,-45.3,-71.8,2015-04-02,275.266327,273.295319,8.254624,0.094248,40.065796
2,0,-45.3,-71.8,2015-04-03,277.224976,274.257233,0.661398,0.155754,34.717327
3,0,-45.3,-71.8,2015-04-04,276.767731,272.226715,0.002491,0.220681,29.512220
4,0,-45.3,-71.8,2015-04-05,278.288666,272.367340,0.000435,0.303918,25.085823


In [42]:
era5_columns = [
    "era5_point_id",
    "sample_date",
    "t2m_K",
    "d2m_K",
    "precip_mm",
    "VPD_kPa",
    "API_mm",
]

sample_era5 = samples.merge(
    era5_daily[era5_columns],
    on=[
        "era5_point_id",
        "sample_date",
    ],
    how="left",
    validate="many_to_one",
)

print(
    "Samples:",
    len(samples),
)

print(
    "Joined:",
    len(sample_era5),
)

assert len(sample_era5) == len(samples)

display(sample_era5.head())

Samples: 3514643
Joined: 3514643


,sample_id,station_id,sample_date,latitude,longitude,era5_lat,era5_lon,era5_point_id,t2m_K,d2m_K,precip_mm,VPD_kPa,API_mm
0,0,AMMA-CATCH__Banizoumbou,2015-04-01,13.5325,2.6604,13.5,2.7,88,302.260223,266.934662,0.003428,3.647409,0.003428
1,1,AMMA-CATCH__Banizoumbou,2015-04-02,13.5325,2.6604,13.5,2.7,88,303.475311,274.992432,0.000855,3.625161,0.003769
2,2,AMMA-CATCH__Banizoumbou,2015-04-03,13.5325,2.6604,13.5,2.7,88,303.875702,262.135803,0.001694,4.159380,0.004898
3,3,AMMA-CATCH__Banizoumbou,2015-04-04,13.5325,2.6604,13.5,2.7,88,304.162445,270.334320,0.000852,3.999373,0.005016
4,4,AMMA-CATCH__Banizoumbou,2015-04-05,13.5325,2.6604,13.5,2.7,88,305.442230,279.767975,0.000435,3.857983,0.004698


In [43]:
era5_feature_columns = [
    "t2m_K",
    "d2m_K",
    "precip_mm",
    "VPD_kPa",
    "API_mm",
]

missing_report = (
    sample_era5[era5_feature_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_percent")
    .to_frame()
)

display(missing_report)

,missing_percent
t2m_K,0.718651
d2m_K,0.718651
VPD_kPa,0.718651
precip_mm,0.053462
API_mm,0.053462


In [44]:
print("Temperature:")
print(
    sample_era5["t2m_K"].describe()
)

print("\nDewpoint:")
print(
    sample_era5["d2m_K"].describe()
)

print("\nPrecipitation:")
print(
    sample_era5["precip_mm"].describe()
)

print("\nVPD:")
print(
    sample_era5["VPD_kPa"].describe()
)

print("\nAPI:")
print(
    sample_era5["API_mm"].describe()
)

Temperature:
count    3.489385e+06
mean     2.869109e+02
std      8.611258e+00
min      2.417577e+02
25%      2.803239e+02
50%      2.869965e+02
75%      2.935186e+02
max      3.178133e+02
Name: t2m_K, dtype: float64

Dewpoint:
count    3.489385e+06
mean     2.776003e+02
std      8.781796e+00
min      2.385901e+02
25%      2.713052e+02
50%      2.771320e+02
75%      2.836563e+02
max      3.002794e+02
Name: d2m_K, dtype: float64

Precipitation:
count    3.512764e+06
mean     2.057615e+00
std      5.322256e+00
min     -4.841422e-05
25%      1.296369e-03
50%      1.043417e-01
75%      1.526016e+00
max      3.707842e+02
Name: precip_mm, dtype: float64

VPD:
count    3.489385e+06
mean     8.078889e-01
std      7.143804e-01
min      0.000000e+00
25%      2.948101e-01
50%      5.851674e-01
75%      1.103277e+00
max      8.587221e+00
Name: VPD_kPa, dtype: float64

API:
count    3.512764e+06
mean     1.357708e+01
std      1.591311e+01
min      0.000000e+00
25%      3.204575e+00
50%      8.78932

In [45]:
ERA5_SAMPLE_OUTPUT = (
    OUTPUT_DIR
    / "era5_sample_features.parquet"
)

sample_era5.to_parquet(
    ERA5_SAMPLE_OUTPUT,
    index=False,
)

print(
    f"Saved {len(sample_era5):,} sample records to:"
)
print(ERA5_SAMPLE_OUTPUT)

Saved 3,514,643 sample records to:
Data/Global/era5_backend/era5_sample_features.parquet


In [46]:
era5_compiled = sample_era5[
    [
        "sample_id",
        "station_id",
        "sample_date",
        "era5_point_id",
        "era5_lat",
        "era5_lon",
        "t2m_K",
        "d2m_K",
        "precip_mm",
        "VPD_kPa",
        "API_mm",
    ]
].copy()

ERA5_COMPILED_OUTPUT = (
    OUTPUT_DIR
    / "era5_by_sample.parquet"
)

era5_compiled.to_parquet(
    ERA5_COMPILED_OUTPUT,
    index=False,
)

print("Saved:", ERA5_COMPILED_OUTPUT)

Saved: Data/Global/era5_backend/era5_by_sample.parquet


In [47]:
print(
    f"ISMN station/date samples: "
    f"{len(samples):,}"
)

print(
    f"Unique stations: "
    f"{samples['station_id'].nunique():,}"
)

print(
    f"Unique ERA5 grid points: "
    f"{samples['era5_point_id'].nunique():,}"
)

compression = (
    len(samples)
    / samples["era5_point_id"].nunique()
)

print(
    f"\nAverage ISMN samples represented "
    f"by each ERA5 point: {compression:.1f}"
)

ISMN station/date samples: 3,514,643
Unique stations: 2,210
Unique ERA5 grid points: 1,489

Average ISMN samples represented by each ERA5 point: 2360.4


In [48]:
sample_era5["year"] = sample_era5["sample_date"].dt.year

year_qc = (
    sample_era5
    .groupby("year")
    .agg(
        samples=("sample_id", "size"),
        missing_precip=("precip_mm", lambda x: x.isna().sum()),
        missing_vpd=("VPD_kPa", lambda x: x.isna().sum()),
        missing_api=("API_mm", lambda x: x.isna().sum()),
    )
    .reset_index()
)

display(year_qc)

,year,samples,missing_precip,missing_vpd,missing_api
0,2015,329903,0,2812,0
1,2016,394790,0,3121,0
2,2017,369953,0,2077,0
3,2018,344806,0,778,0
4,2019,308679,0,826,0
5,2020,269853,0,1137,0
6,2021,259625,0,1034,0
7,2022,260490,0,2162,0
8,2023,233307,0,2572,0
9,2024,264946,0,2920,0


In [49]:
summary = {
    "n_samples": int(len(sample_era5)),
    "n_stations": int(sample_era5["station_id"].nunique()),
    "n_era5_grid_points": int(sample_era5["era5_point_id"].nunique()),
    "min_date": str(sample_era5["sample_date"].min().date()),
    "max_date": str(sample_era5["sample_date"].max().date()),
    "api_decay": API_DECAY,
    "era5_resolution_deg": ERA5_GRID_DEG,
    "variables": [
        "t2m_K",
        "d2m_K",
        "precip_mm",
        "VPD_kPa",
        "API_mm",
    ],
}

with open(
    OUTPUT_DIR / "era5_stage_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )

print(json.dumps(summary, indent=2))

{
  "n_samples": 3514643,
  "n_stations": 2210,
  "n_era5_grid_points": 1489,
  "min_date": "2015-04-01",
  "max_date": "2026-10-03",
  "api_decay": 0.85,
  "era5_resolution_deg": 0.1,
  "variables": [
    "t2m_K",
    "d2m_K",
    "precip_mm",
    "VPD_kPa",
    "API_mm"
  ]
}
